# 02 - Cloud YOLO Training and Evaluation



This notebook consumes the immutable dataset prepared by `01_cloud_yolo_dataset_prep.ipynb`. It does not download labels, alter class mappings, or rebuild splits.



Workflow:



1. Point `WORKSPACE` at notebook 1's prepared workspace.

2. Verify `dataset.yaml`, `split_manifest.json`, `prep_summary.json`, and split contents.

3. Select simple or advanced Ultralytics training settings.

4. Train a new run or resume from `last.pt`.

5. Evaluate validation and test splits.

6. Upload versioned run artifacts to GCS.



Use the `Python (yolo-cloud)` kernel created by `install_cloud_workstation.sh`. See the [Ultralytics train settings](https://docs.ultralytics.com/modes/train/#train-settings) before changing advanced parameters.

In [ ]:
# %pip install -U "ultralytics>=8.4.92" pyyaml



import hashlib

import json

import os

import shutil

import subprocess

import sys

from datetime import datetime, timezone

from importlib.metadata import version

from pathlib import Path



import torch

import yaml

from ultralytics import YOLO



PROJECT_NAME = "serdp_yolo_v0"

WORKSPACE = Path.cwd() / "workspace" / PROJECT_NAME

DATASET_YAML = WORKSPACE / "dataset.yaml"

SPLIT_MANIFEST = WORKSPACE / "split_manifest.json"

PREP_SUMMARY = WORKSPACE / "prep_summary.json"

RUNS_DIR = WORKSPACE / "runs"



GCP_PROJECT_ID = "REPLACE_WITH_GCP_PROJECT_ID"

OUTPUT_BUCKET_URI = "gs://REPLACE_BUCKET/REPLACE_PATH/training-runs"



MODEL_NAME = "yolo26m.pt"

EPOCHS = 100

IMAGE_SIZE = 640

BATCH_SIZE = 16

DEVICE = 0 if torch.cuda.is_available() else "cpu"

RUN_DATETIME_UTC = datetime.now(timezone.utc)

RUN_NAME = f"{PROJECT_NAME}_{RUN_DATETIME_UTC:%Y%m%dT%H%M%SZ}"



USE_ADVANCED_TRAINING = False

RUN_TRAINING = True

RESUME_FROM = None  # Example: RUNS_DIR / "prior_run" / "weights" / "last.pt"



RUNS_DIR.mkdir(parents=True, exist_ok=True)

print({

    "workspace": str(WORKSPACE),

    "run_name": RUN_NAME,

    "python": sys.version.split()[0],

    "torch": torch.__version__,

    "cuda_available": torch.cuda.is_available(),

    "device": DEVICE,

})

## 1. Verify the dataset-preparation handoff



This gate fails before GPU work when contract files are missing, content hashes changed, configured classes disagree, or any split has unmatched images and labels. If it fails, return to notebook 1 instead of repairing derived files here.

In [ ]:
def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for block in iter(lambda: file.read(1024 * 1024), b""):

            digest.update(block)

    return digest.hexdigest()





for required_path in (DATASET_YAML, SPLIT_MANIFEST, PREP_SUMMARY):

    if not required_path.is_file():

        raise FileNotFoundError(f"Run notebook 1 first; missing handoff artifact: {required_path}")



dataset_config = yaml.safe_load(DATASET_YAML.read_text(encoding="utf-8"))

split_manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))

prep_summary = json.loads(PREP_SUMMARY.read_text(encoding="utf-8"))



if sha256_file(DATASET_YAML) != prep_summary["dataset_yaml_sha256"]:

    raise ValueError("dataset.yaml changed after preparation. Rerun notebook 1.")

if sha256_file(SPLIT_MANIFEST) != prep_summary["split_manifest_sha256"]:

    raise ValueError("split_manifest.json changed after preparation. Rerun notebook 1.")



configured_names = dataset_config.get("names", {})

dataset_names = list(configured_names.values()) if isinstance(configured_names, dict) else list(configured_names)

if dataset_names != prep_summary["classes"]:

    raise ValueError(f"Class mismatch: YAML={dataset_names}, prep={prep_summary['classes']}")



dataset_root = Path(dataset_config["path"])

split_counts = {}

for split in ("train", "val", "test"):

    image_dir = dataset_root / "images" / split

    label_dir = dataset_root / "labels" / split

    image_stems = {path.stem for path in image_dir.iterdir() if path.is_file()}

    label_stems = {path.stem for path in label_dir.glob("*.txt")}

    if image_stems != label_stems:

        raise ValueError(

            f"{split} pairing changed: missing labels={sorted(image_stems - label_stems)[:10]}, "

            f"orphan labels={sorted(label_stems - image_stems)[:10]}"

        )

    split_counts[split] = len(image_stems)



if split_counts != prep_summary["split_counts"]:

    raise ValueError(f"Split counts changed: current={split_counts}, prepared={prep_summary['split_counts']}")



print(json.dumps({

    "prep_name": prep_summary["prep_name"],

    "seed": prep_summary["seed"],

    "classes": dataset_names,

    "split_counts": split_counts,

    "dataset_yaml_sha256": prep_summary["dataset_yaml_sha256"],

}, indent=2))

## 2. Simple training baseline



Leave `USE_ADVANCED_TRAINING = False` for the first run. These settings use Ultralytics defaults except for the core project, device, output, and reproducibility controls. Training uses notebook 1's recorded split seed.

In [ ]:
SIMPLE_TRAIN_ARGS = {

    "data": str(DATASET_YAML),

    "epochs": EPOCHS,

    "imgsz": IMAGE_SIZE,

    "batch": BATCH_SIZE,

    "device": DEVICE,

    "project": str(RUNS_DIR),

    "name": RUN_NAME,

    "exist_ok": False,

    "pretrained": True,

    "seed": prep_summary["seed"],

    "deterministic": True,

}

print(json.dumps(SIMPLE_TRAIN_ARGS, indent=2))

## 3. Advanced training options



Set `USE_ADVANCED_TRAINING = True` to enable runtime, optimizer, augmentation, loss, and fine-tuning controls modeled on the ESA and urchin notebooks. Keep transformations physically plausible for the project. The complete and current parameter reference is in the [Ultralytics train settings](https://docs.ultralytics.com/modes/train/#train-settings).



Set `RESUME_FROM` to an interrupted run's `last.pt` to restore its optimizer, scheduler, and epoch state. Resume uses the checkpoint's saved arguments rather than the dictionaries below.

In [ ]:
FREEZE = None

ADVANCED_TRAIN_ARGS = {

    **SIMPLE_TRAIN_ARGS,

    "patience": 45,

    "save_period": 10,

    "workers": 4 if os.name == "nt" else 8,

    "cache": False,

    "amp": torch.cuda.is_available(),

    "plots": True,

    "optimizer": "AdamW",

    "lr0": 5e-4,

    "lrf": 0.01,

    "weight_decay": 0.001,

    "warmup_epochs": 3.0,

    "cos_lr": True,

    "augment": True,

    "degrees": 0.0,

    "translate": 0.1,

    "scale": 0.5,

    "fliplr": 0.5,

    "flipud": 0.0,

    "mosaic": 1.0,

    "close_mosaic": 15,

    "mixup": 0.0,

    "box": 7.5,

    "cls": 0.5,

    "dfl": 1.5,

}

if FREEZE is not None:

    ADVANCED_TRAIN_ARGS["freeze"] = FREEZE



selected_train_args = ADVANCED_TRAIN_ARGS if USE_ADVANCED_TRAINING else SIMPLE_TRAIN_ARGS

training_mode = "advanced" if USE_ADVANCED_TRAINING else "simple"

print(f"Selected training mode: {training_mode}")

print(json.dumps(selected_train_args, indent=2))



run_metadata = {

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "project_name": PROJECT_NAME,

    "run_name": RUN_NAME,

    "prep_summary": prep_summary,

    "training_mode": training_mode,

    "training_args": selected_train_args,

    "versions": {

        "python": sys.version.split()[0],

        "torch": torch.__version__,

        "ultralytics": version("ultralytics"),

    },

}

(WORKSPACE / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")



if RESUME_FROM is not None:

    resume_path = Path(RESUME_FROM)

    if not resume_path.exists():

        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")

    model = YOLO(str(resume_path))

    training_results = model.train(resume=True)

elif RUN_TRAINING:

    model = YOLO(MODEL_NAME)

    training_results = model.train(**selected_train_args)

else:

    training_results = None

    print("RUN_TRAINING is False; training was skipped.")



run_dir = Path(training_results.save_dir) if training_results is not None else RUNS_DIR / RUN_NAME

print(f"Run directory: {run_dir}")

## 4. Evaluate validation and test splits



Evaluation prefers `best.pt` and falls back to `last.pt`. Validation guides model selection; the held-out test split should be used for final reporting rather than repeated tuning.

In [ ]:
best_weights = run_dir / "weights" / "best.pt"

last_weights = run_dir / "weights" / "last.pt"

weights_for_evaluation = best_weights if best_weights.exists() else last_weights

if not weights_for_evaluation.exists():

    raise FileNotFoundError(f"No checkpoint found under: {run_dir / 'weights'}")



evaluation_model = YOLO(str(weights_for_evaluation))

evaluation_summary = {"weights": str(weights_for_evaluation)}



for split in ("val", "test"):

    if split_counts[split] == 0:

        evaluation_summary[split] = None

        continue

    metrics = evaluation_model.val(

        data=str(DATASET_YAML),

        split=split,

        imgsz=IMAGE_SIZE,

        batch=BATCH_SIZE,

        device=DEVICE,

        plots=True,

        project=str(run_dir),

        name=f"{split}_evaluation",

    )

    evaluation_summary[split] = {

        "precision": float(metrics.box.mp),

        "recall": float(metrics.box.mr),

        "map50": float(metrics.box.map50),

        "map50_95": float(metrics.box.map),

    }



metrics_path = run_dir / "evaluation_metrics.json"

metrics_path.write_text(json.dumps(evaluation_summary, indent=2), encoding="utf-8")

print(json.dumps(evaluation_summary, indent=2))

## 5. Upload versioned training artifacts to GCS



This publishes the run directory together with the three preparation contract files and run metadata. It does not upload the local dataset images. Set `UPLOAD_ARTIFACTS = False` when testing locally.

In [ ]:
UPLOAD_ARTIFACTS = True



def run_command(arguments: list[str], *, capture: bool = False) -> subprocess.CompletedProcess[str]:

    print("Running:", subprocess.list2cmdline(arguments))

    return subprocess.run(arguments, check=True, text=True, capture_output=capture)





if UPLOAD_ARTIFACTS:

    if shutil.which("gcloud") is None:

        raise RuntimeError("gcloud was not found. Run the workstation setup and restart JupyterLab.")

    if "REPLACE_WITH" in GCP_PROJECT_ID or "REPLACE_" in OUTPUT_BUCKET_URI:

        raise ValueError("Set GCP_PROJECT_ID and OUTPUT_BUCKET_URI before uploading.")



    active_account = run_command(

        ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=value(account)"],

        capture=True,

    ).stdout.strip()

    if not active_account:

        raise RuntimeError("No active gcloud account. Authenticate in a JupyterLab terminal.")



    run_command(["gcloud", "config", "set", "project", GCP_PROJECT_ID])

    for artifact in (DATASET_YAML, SPLIT_MANIFEST, PREP_SUMMARY, WORKSPACE / "run_metadata.json"):

        shutil.copy2(artifact, run_dir / artifact.name)



    artifact_uri = f"{OUTPUT_BUCKET_URI.rstrip('/')}/{RUN_NAME}"

    run_command(["gcloud", "storage", "rsync", "--recursive", str(run_dir), artifact_uri])

    print(f"Team artifact URI: {artifact_uri}")

    print(f"Best model URI: {artifact_uri}/weights/{weights_for_evaluation.name}")

else:

    print("UPLOAD_ARTIFACTS is False; local run artifacts were not uploaded.")

## 6. Run checklist



- Treat notebook 1's split as immutable for a training experiment.

- Start with simple mode and preserve its metrics as the baseline.

- Change one justified parameter group at a time in advanced mode.

- Use validation for tuning and reserve test results for final comparison.

- Resume only from `last.pt`; use `best.pt` for evaluation and deployment.

- Preserve `dataset.yaml`, `split_manifest.json`, `prep_summary.json`, `run_metadata.json`, metrics, plots, and weights together.

- If class definitions or labels change, rerun notebook 1 and create a new preparation identity.